# SPY 2DTE Momentum-Trigger Strategy

**Entry signal** (each trading day):
- Anchor: SPY's 9:30 ET regular-session open price.
- Starting at **10:34 ET (7:34 PT)**, check each 1-min candle in sequence (up to 5 attempts):
  - If the **previous** 1-min candle's close is ≥ open + $0.15 → buy a 1-step ITM 2DTE **call**
  - If the previous close is ≤ open − $0.15 → buy a 1-step ITM 2DTE **put**
  - **RSI(14)** on same-day SPY 1-min closes must be in **[30, 70]** at the previous candle.
  - If trigger doesn't fire or RSI is outside the band, advance one minute.
  - After 5 attempts without entry, give up for the day.
- Entry fills at the qualifying minute's option **open** (single leg long).

**Position:** 1-step ITM, 2 trading days to expiration (across weekends — Mon→Wed, Tue→Thu, Wed→Fri, Thu→Mon, Fri→Tue). Strikes are $1 apart at ATM.

**Exits (swept):**
- **Profit target:** 2% gross (mid-to-mid), 2% net (after fees/slippage), 3% gross, or 3% net.
- **Stop loss:** one of {20%, 25%, 30%, 35%, 40%, 45%, 50%} *or* a time-stop of {60, 90, 120} min from entry.
- **Hard close:** 3:55 PM ET fallback to avoid overnight risk.

**Sweep grid:** 4 PTs × 10 SLs = **40 configs**.

**Account:** $1,500 starting balance, $50k max capital per trade, $0.85/contract commission (IBKR-calibrated).


## 1. Setup

In [ ]:
import os, sys
REPO = '/content/iron-condor'
BRANCH = 'claude/spy-options-trading-bot-ri4Ms'
if not os.path.isdir(REPO):
    !git clone --quiet -b {BRANCH} https://github.com/coolin815/iron-condor.git {REPO}
else:
    !cd {REPO} && git pull --quiet
SRC = REPO + '/src'
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH', '')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.chdir(REPO)
print('OK')

## 2. Paste your Polygon key

In [ ]:
import os, getpass
os.environ['POLYGON_API_KEY'] = getpass.getpass('Polygon API key: ')
print('Key loaded:', len(os.environ['POLYGON_API_KEY']), 'chars')

## 3. Smoke test — one recent day
Confirms the trigger pipeline runs end-to-end and the 2DTE contract resolves. Output is a single TradeResult (or `no_signal`/`no_data` if today didn't trigger).

In [ ]:
!python -m iron_condor.cli --smoke -v

## 4. Sweep — past 7 days
~1–2 min. ~5 trading days. With only ~5 entries (one per day max), numbers will be noisy — we're checking that the pipeline works.

In [ ]:
!python -m iron_condor.cli --days 7 --sweep

## 4b. Sweep — past 30 days
~5 min. ~21 trading days. First read on actual edge.

In [ ]:
!python -m iron_condor.cli --days 30 --sweep

## 4c. Sweep — past 60 days
~10 min. ~42 trading days.

In [ ]:
!python -m iron_condor.cli --days 60 --sweep

## 4d. Sweep — past 365 days (optional)
Full year. Run only if 60-day looked promising.

In [ ]:
!python -m iron_condor.cli --days 365 --sweep

## 4e. Diagnostic — see which days triggered
Runs one config (10% PT, 30% SL) over the past 7 days with verbose logging so we can see entry decisions per day.

In [ ]:
!python -m iron_condor.cli --days 7 -v 2>&1 | grep -E '(attempt|ENTRY|no signal)'

## 5. Top configs
Reads `results/sweep_summary.csv` written by whichever sweep cell you last ran.

In [ ]:
import pandas as pd
summary = pd.read_csv('results/sweep_summary.csv')
summary.head(30)

## 6. PT × SL grid
Heat-map view of return / win rate by profit-target and stop-loss config.

In [ ]:
import re
rows = []
for _, r in summary.iterrows():
    m = re.match(r'pt(\d+)([gn])\|sl(\d+)(min)?', r['config'])
    if not m:
        continue
    pt = int(m.group(1))
    mode = 'gross' if m.group(2) == 'g' else 'net'
    sl_val = int(m.group(3))
    sl_kind = 'min' if m.group(4) == 'min' else 'pct'
    rows.append({
        'pt_label': f'{pt}%{mode[0]}',
        'sl_label': f'{sl_val}{sl_kind}',
        'return': r['total_return_pct'],
        'win_rate': r['win_rate'],
        'trades': r['n_trades'],
    })
grid = pd.DataFrame(rows)

print('=== Total return % by (PT, SL) ===')
print(grid.pivot(index='pt_label', columns='sl_label', values='return').round(1))
print('\n=== Win rate by (PT, SL) ===')
print(grid.pivot(index='pt_label', columns='sl_label', values='win_rate').round(2))
print('\n=== Trade count by (PT, SL) ===')
print(grid.pivot(index='pt_label', columns='sl_label', values='trades').round(0).astype('Int64'))

## 7. Top config breakdown — call vs put, exit reasons

In [ ]:
trades = pd.read_csv('results/sweep_trades.csv')
top_cfg = summary.iloc[0]['config']
sub = trades[trades['config'] == top_cfg]
taken = sub[sub['exit_reason'].isin(['profit', 'stop', 'time_stop', 'hard_close'])]
print(f'Top config: {top_cfg}')
print(f'Total days: {len(sub)}, taken trades: {len(taken)}')
print('\n=== Call vs put performance ===')
if not taken.empty and 'right' in taken.columns:
    print(taken.groupby('right').agg(
        trades=('net_pnl', 'count'),
        win_rate=('net_pnl', lambda s: (s > 0).mean()),
        avg_pnl=('net_pnl', 'mean'),
        total_pnl=('net_pnl', 'sum'),
    ).round(2))
print('\n=== Exit reasons ===')
print(sub['exit_reason'].value_counts())
print('\n=== Hold times (minutes) — taken trades only ===')
if not taken.empty and 'minutes_held' in taken.columns:
    print(taken['minutes_held'].describe().round(1))

## 8. Equity curve — top 3 configs

In [ ]:
import matplotlib.pyplot as plt
top3 = summary.head(3)['config'].tolist()
fig, ax = plt.subplots(figsize=(11, 5))
for cfg in top3:
    sub = trades[trades['config'] == cfg].sort_values('day')
    ax.plot(pd.to_datetime(sub['day']), sub['balance_after'], label=cfg, linewidth=1.5)
ax.axhline(1500, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Date'); ax.set_ylabel('Balance ($)')
ax.set_title('2DTE momentum trigger — top 3 equity curves')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 9. Per-month breakdown
Splits the trade log by calendar month to see if any month carries the result.

In [ ]:
trades_all = pd.read_csv('results/sweep_trades.csv')
trades_all['day'] = pd.to_datetime(trades_all['day'])
trades_all['month'] = trades_all['day'].dt.to_period('M').astype(str)
taken_all = trades_all[trades_all['exit_reason'].isin(['profit', 'stop', 'time_stop', 'hard_close'])]

print('=== Per-month aggregate (all configs averaged) ===')
monthly = taken_all.groupby('month').agg(
    trades=('net_pnl', 'count'),
    win_rate=('net_pnl', lambda s: (s > 0).mean()),
    avg_pnl=('net_pnl', 'mean'),
    total_pnl=('net_pnl', 'sum'),
).round(2)
print(monthly.to_string())

print(f'\n=== Top config ({summary.iloc[0]["config"]}) per month ===')
top_cfg = summary.iloc[0]['config']
top_taken = taken_all[taken_all['config'] == top_cfg]
if not top_taken.empty:
    top_monthly = top_taken.groupby('month').agg(
        trades=('net_pnl', 'count'),
        win_rate=('net_pnl', lambda s: (s > 0).mean()),
        avg_pnl=('net_pnl', 'mean'),
        total_pnl=('net_pnl', 'sum'),
    ).round(2)
    print(top_monthly.to_string())
else:
    print('(no taken trades for top config)')